# [7] AttentiveSentinel Model suggested by [@gtend](https://github.com/gtend) (유채민)

# Imports

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import re

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
# project name
PROJECT_NAME = "7_attentive_sentinel"

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
base_model_id = "klue/bert-base"

In [ ]:
base_model_id = "monologg/koelectra-base-v3-discriminator"

In [ ]:
class AttentiveSentinel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        self.bert = AutoModel.from_pretrained(base_model_id)
        self.embedding_dim = self.bert.config.hidden_size

        self.attention_net = nn.Sequential(
            nn.Linear(self.embedding_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        self.classifier = nn.Linear(self.embedding_dim, num_classes)

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, sentences: list):
        tokenized_inputs = self.tokenizer(
            sentences,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            bert_output = self.bert(**tokenized_inputs)

        sentence_embeddings = self._mean_pooling(bert_output, tokenized_inputs['attention_mask'])

        attention_raw_scores = self.attention_net(sentence_embeddings)
        attention_weights = F.softmax(attention_raw_scores, dim=0)

        document_vector = torch.sum(sentence_embeddings * attention_weights, dim=0)

        logits = self.classifier(document_vector)

        return logits, attention_weights

In [ ]:
# ==============================================================================
# 5. 모델 정의 (Model Definition)
# ==============================================================================
class SentenceAttentionModel(nn.Module):
    def __init__(self, num_classes=1):
        super(SentenceAttentionModel, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.embedding_dim = self.bert.config.hidden_size
        self.attention_net = nn.Sequential(nn.Linear(self.embedding_dim, 256), nn.Tanh(), nn.Linear(256, 1))
        self.dropout_rate = 0.3
        self.dropout = nn.Dropout(self.dropout_rate)
        # self.classifier = nn.Linear(self.embedding_dim, num_classes)
        
        self.hidden_size = 256 # 중간 은닉층의 크기

        self.classifier = nn.Sequential(
            nn.Linear(self.embedding_dim, self.hidden_size), # 첫 번째 레이어: 문서벡터 -> 은닉층
            nn.ReLU(),                                  # 비선형성을 더해줄 활성화 함수
            nn.Dropout(self.dropout_rate),                   # 과적합 방지를 위한 드롭아웃
            nn.Linear(self.hidden_size, num_classes)         # 두 번째 레이어: 은닉층 -> 최종 출력
        )
        
    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def forward(self, texts: list):
        batch_logits = []
        for text in texts:
            sentences = split_sentences_without_library(text)
            if not sentences: continue
            
            tokenized = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
            
            # BERT 통과 시에는 그래디언트 계산을 하지 않음 (메모리 및 속도 향상)
            with torch.no_grad():
                bert_output = self.bert(**tokenized)
            
            sentence_embeddings = self._mean_pooling(bert_output, tokenized['attention_mask'])
            
            attention_scores = F.softmax(self.attention_net(sentence_embeddings), dim=0)
            document_vector = torch.sum(sentence_embeddings * attention_scores, dim=0)
            
            # document_vector_with_dropout = self.dropout(document_vector)
            # logits = self.classifier(document_vector)
            
            logits = self.classifier(document_vector)
            batch_logits.append(logits)
        
        return torch.stack(batch_logits) if batch_logits else None

In [ ]:
model = AttentiveSentinel()
try:
    from safetensors.torch import load_file
    state_dict = load_file(f"./models/{PROJECT_NAME}_last/model.safetensors")
    model.load_state_dict(state_dict)
except Exception:
    pass
model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

### Utils

In [ ]:
def split_sentences_without_library(text):
    sentences = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sentences if s]

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = AdamW(trainable_params, lr=LEARNING_RATE) # lr=5e-5 등 fine-tuning에 맞는 값 추천

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("학습 준비 완료.")

# ==============================================================================
# 7. 학습 및 검증 루프 (Training & Validation Loop)
# ==============================================================================
best_f1 = 0
print("\n학습 시작!")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    all_train_preds, all_train_labels = [], []
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        texts = batch['texts']
        labels = batch['labels'].to(DEVICE)
        
        logits = model(texts)
        if logits is None: continue
        
        loss = criterion(logits, labels)
        
        total_train_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
        preds = torch.argmax(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'Train Loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')

    # --- 검증 모드 ---
    model.eval()
    total_val_loss = 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
        for batch in progress_bar:
            texts = batch['texts']
            labels = batch['labels'].to(DEVICE)

            logits = model(texts)
            if logits is None: continue
            
            loss = criterion(logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
    
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f}")
    print("-" * 80)
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "attmodelV6.pt")
        print("✅ 모델 저장 완료 (AUC 갱신)")

In [ ]:
model.eval()
total_val_loss = 0
all_val_preds, all_val_probs, all_val_labels = [], [], []

with torch.no_grad():
    progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
    for batch in progress_bar:
        texts = batch['texts']
        labels = batch['labels'].to(DEVICE)

        logits = model(texts)
        if logits is None: continue

        # loss = criterion(logits, labels)
        # total_val_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        # preds = F.softmax(logits, dim=1)

        
        probs = F.softmax(logits, dim=1)
        # print(probs)
        # max_probs = torch.max(probs, dim=1).values
        probs = probs[:, 1]
        
        # print("max : ", probs)
        # probs = probs[0]
        # if probs[0] > probs[1]:
        #     final = probs[1]
        # else :
        #     final = probs[0]


        # index = torch.argmax(preds, dim=1)
        
        # probs = preds[index]
        
        # print(probs)
        # print(logits.shape)
        # print(preds)
        all_val_preds.extend(preds.cpu().numpy())        
        all_val_probs.extend(probs.cpu().numpy())
        all_val_labels.extend(labels.cpu().numpy())